# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets and their fields using their `@id` values.

We'll enumerate available record sets in the dataset, then explore their fields (columns).

In [ ]:
# List all record sets by their @id and name (if available)
record_sets = []
if hasattr(dataset, 'record_sets'):
    for record_set in dataset.record_sets:
        rid = record_set['@id']
        name = record_set.get('name', '(no name)')
        print(f"RecordSet @id: {rid} | name: {name}")
        record_sets.append(rid)
else:
    # Fallback for mlcroissant>=0.8 which exposes get_record_sets()
    for record_set in dataset.get_record_sets():
        rid = record_set['@id']
        name = record_set.get('name', '(no name)')
        print(f"RecordSet @id: {rid} | name: {name}")
        record_sets.append(rid)

if not record_sets:
    print('No record sets found in the metadata; loading records will attempt anyway.')

In [ ]:
# If at least one record set is present, show some records and the field @ids
if record_sets:
    record_set_id = record_sets[0]

    print(f"\nSample records for RecordSet @id: {record_set_id}")
    for i, row in enumerate(dataset.records(record_set=record_set_id)):
        print(json.dumps(row, indent=2))
        if i >= 2:  # Show only the first 3 records
            break
    # Print available field (column) names
    print("\nAvailable fields (columns) by @id:")
    for field in dataset.get_fields(record_set=record_set_id):
        print(f"Field @id: {field['@id']}, name: {field.get('name', '')}")
else:
    # If record sets not explicitly listed, try to iterate records from default record set
    print("No record sets found; attempting to iterate records without specifying record_set.")
    for i, row in enumerate(dataset.records()):
        print(json.dumps(row, indent=2))
        if i >= 2:
            break

## 3. Data Extraction
Load data from the selected record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract from the first available record set, if any were found
if record_sets:
    dataframes = {}
    for rs_id in record_sets:
        # Load all records for the record set
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rs_id])} records for RecordSet @id {rs_id}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(2))
    # Choose the first record set for the rest of the notebook
    df_key = record_sets[0]
else:
    # No record set IDs found; fallback: try without specifying record_set
    df = pd.DataFrame(list(dataset.records()))
    print(f"Loaded {len(df)} records; columns available: {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records by a numeric field, normalizing, and grouping using field `@id`s.

For demonstration, we'll use a numeric field likely present in mass spectrometry protein datasets, such as `coverage_percent`, `peptide_count`, or `mw` (molecular weight). All columns are referenced using their `@id` as obtained from the schema.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

# Use the first DataFrame extracted, and try to find a likely numeric field for analysis
if record_sets:
    df = dataframes[df_key]
    columns = df.columns.tolist()
    print("Columns detected in loaded DataFrame:")
    print(columns)
    # Try to pick a numeric field for demonstration
    possible_numeric_ids = [c for c in columns if any(keyword in c.lower() for keyword in ['coverage', 'count', 'mw', 'abundance', 'intensity', 'peptide'])]
    numeric_field_id = possible_numeric_ids[0] if possible_numeric_ids else columns[0]
    print(f"Using numeric field @id for EDA: {numeric_field_id}")

    # Convert the field to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = float(df[numeric_field_id].mean() or 10)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean value): {len(filtered_df)} records found.")

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another likely attribute field, e.g. 'accession' or 'description'
    possible_group_fields = [c for c in columns if any(keyword in c.lower() for keyword in ['accession', 'description', 'sample', 'group', 'category'])]
    group_field_id = possible_group_fields[0] if possible_group_fields else columns[0]

    print(f"Grouping data by field @id: {group_field_id}")
    if group_field_id in filtered_df.columns:
        group_stats = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().head()
        print(group_stats)
else:
    print('No DataFrame to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the record set. For example, a histogram of a numeric field or boxplot per group.

In [ ]:
import matplotlib.pyplot as plt

if record_sets:
    # Plot histogram of the numeric field
    plt.figure(figsize=(7,4))
    df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot of numeric_field by group_field_id (if they are not the same)
    if group_field_id != numeric_field_id and pd.api.types.is_string_dtype(df[group_field_id]):
        plt.figure(figsize=(10,4))
        # Plot only groups with reasonable sample size
        to_plot = df.groupby(group_field_id)[numeric_field_id].count().sort_values(ascending=False).head(10).index
        df[df[group_field_id].isin(to_plot)].boxplot(column=numeric_field_id, by=group_field_id, rot=60)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric fields available to plot.')

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library. We:
- Loaded metadata and identified record sets and fields by their `@id`.
- Extracted tabular data using those `@id`s.
- Performed basic EDA such as filtering, normalization, and grouping by key attributes.
- Visualized numeric distributions to facilitate further biological or computational analysis.

All dataset entities and fields should be referenced using their `@id` as per Croissant best-practices, ensuring schema-consistent processing. For advanced analysis (e.g., protein clustering, biomarker search), refer to the Croissant schema and dataset documentation for further field semantics.